# Phase 0 — training timing calibration on T4

Measures what `dreaddevelopment`'s training script actually costs on Kaggle's T4, so the
epoch budget for later phases is measured rather than guessed. Their reference is
*"about three hours on one 4090 for 16 epochs at 384"*; a T4 is materially slower and the
session cap is 12 h, so this decides the shape of every later run.

Also checks two hardware issues found by reading the script:

- it autocasts to **bfloat16**, and T4 (sm_75) has no native bf16 — same gap that made the
  P100 unusable earlier. fp16 is timed alongside as the alternative.
- `_find()` walks all of `/kaggle/input`; with the competition mounted that is ~4,407 DICOM
  study directories, so the corpus files are symlinked into the working dir first.

In [ ]:
TRAIN_PY_B64 = (
    '...'
)

import base64, os, pathlib, textwrap, time, json, subprocess, sys

WORK = pathlib.Path('/kaggle/working')
(WORK / 'train_knee.py').write_bytes(base64.b64decode(TRAIN_PY_B64))
print('wrote train_knee.py', (WORK / 'train_knee.py').stat().st_size, 'bytes')

In [ ]:
# Stage the corpus next to the script so _find() never walks the DICOM tree.
import os, pathlib, time
WORK = pathlib.Path('/kaggle/working')
t0 = time.time()
need = ['all_vols.npy','all_masks.npy','all_ids.npy',
        'extra_vols.npy','extra_masks.npy','extra_ids.npy']
found = {}
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if d not in ('train_series','test_series')]
    for n in need:
        if n in files and n not in found:
            found[n] = os.path.join(root, n)
    if len(found) == len(need):
        break
print(f'located {len(found)}/{len(need)} corpus files in {time.time()-t0:.1f}s')
for n, p in found.items():
    print(f'  {n:18s} {os.path.getsize(p)/1e9:6.2f} GB  {p}')
missing = [n for n in need if n not in found]
assert not missing, f'missing corpus files: {missing}'
# NOTE: do NOT stage all_vols.npy locally -- it is 16 GB and /kaggle/working is capped.
# Only the small index files need to be local; the big volumes stay memmapped in place.

In [ ]:
# Locate labels + train.csv, and report the environment.
import os, glob, pandas as pd, torch, subprocess
lab = None
for pat in ('/kaggle/input/*/labels_llm_soft.parquet','/kaggle/input/*/*/labels_llm_soft.parquet'):
    g = glob.glob(pat)
    if g: lab = g[0]; break
print('labels parquet :', lab)
if lab:
    L = pd.read_parquet(lab); print('  shape', L.shape)
tr = glob.glob('/kaggle/input/**/train.csv', recursive=False) or \
     glob.glob('/kaggle/input/*/train.csv')
print('train.csv      :', tr[:1])
print()
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    cc = torch.cuda.get_device_capability(i)
    print(f'  gpu{i}: {torch.cuda.get_device_name(i)} sm_{cc[0]}{cc[1]} '
          f'{torch.cuda.get_device_properties(i).total_memory/2**30:.0f} GiB '
          f'native_bf16={cc >= (8,0)}')
print()
print(subprocess.run(['df','-h','/kaggle/working'],capture_output=True,text=True).stdout)

In [ ]:
# Micro-benchmark: bf16 vs fp16 vs fp32 for the actual backbone at 384, before any training.
import torch, timm, time
ARCH, RES = 'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k', 384
dev = 'cuda'
m = timm.create_model(ARCH, pretrained=False, num_classes=0).to(dev).train()
opt = torch.optim.AdamW(m.parameters(), lr=1e-5)
x = torch.randn(4, 3, RES, RES, device=dev)

def bench(dtype, n=6, scaler=None):
    torch.cuda.synchronize(); ts = []
    for i in range(n):
        t = time.time()
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=dtype, enabled=dtype is not None):
            y = m(x).float().sum()
        if scaler is not None:
            scaler.scale(y).backward(); scaler.step(opt); scaler.update()
        else:
            y.backward(); opt.step()
        torch.cuda.synchronize()
        if i >= 2: ts.append(time.time() - t)   # discard warmup
    return sum(ts)/len(ts)

res = {}
for name, dt, sc in (('bf16', torch.bfloat16, None),
                     ('fp16', torch.float16, torch.amp.GradScaler('cuda')),
                     ('fp32', None, None)):
    try:
        res[name] = bench(dt, scaler=sc)
        print(f'  {name}: {res[name]*1000:7.1f} ms / step (bs=4 @ {RES})')
    except Exception as e:
        print(f'  {name}: FAILED {type(e).__name__}: {e}')
if 'bf16' in res and 'fp16' in res:
    print(f'\n  fp16 is {res["bf16"]/res["fp16"]:.2f}x faster than bf16 on this GPU')
del m, opt, x; torch.cuda.empty_cache()

In [ ]:
# Wiring check on a tiny subset (--smoke), exactly as published apart from paths.
import subprocess, sys, time, glob, os
lab = (glob.glob('/kaggle/input/*/labels_llm_soft.parquet') +
       glob.glob('/kaggle/input/*/*/labels_llm_soft.parquet'))[0]
cmd = [sys.executable, '/kaggle/working/train_knee.py', '--smoke',
       '--arch', 'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k', '--res', '384',
       '--bs', '4', '--k', '4', '--k_eval', '4', '--grad_ckpt',
       '--labels', lab, '--tag', 'smoke']
print(' '.join(cmd), flush=True)
t0 = time.time()
p = subprocess.run(cmd, cwd='/kaggle/working', capture_output=True, text=True)
print(f'--- smoke exit={p.returncode} in {time.time()-t0:.1f}s ---')
print(p.stdout[-4000:])
if p.returncode != 0:
    print('STDERR:', p.stderr[-4000:])

In [ ]:
# One real epoch on a capped subset -> seconds per study, then project a full epoch.
import subprocess, sys, time, glob, json, os
lab = (glob.glob('/kaggle/input/*/labels_llm_soft.parquet') +
       glob.glob('/kaggle/input/*/*/labels_llm_soft.parquet'))[0]
LIMIT = 300
cmd = [sys.executable, '/kaggle/working/train_knee.py',
       '--arch', 'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k', '--res', '384',
       '--epochs', '1', '--bs', '8', '--k', '12', '--k_eval', '24',
       '--grad_ckpt', '--limit', str(LIMIT), '--labels', lab, '--tag', 'timing']
print(' '.join(cmd), flush=True)
t0 = time.time()
p = subprocess.run(cmd, cwd='/kaggle/working', capture_output=True, text=True)
EL = time.time() - t0
print(f'--- exit={p.returncode} in {EL:.1f}s for {LIMIT} studies ---')
print(p.stdout[-5000:])
if p.returncode != 0:
    print('STDERR:', p.stderr[-5000:])
else:
    per = EL / LIMIT
    full = per * 4349
    print(f'\n  {per:.3f} s/study (train+eval, 1 epoch, includes model build + gold eval)')
    print(f'  projected full epoch over 4,349 studies: {full/60:.1f} min')
    for ep in (8, 12, 16):
        print(f'    {ep:2d} epochs -> {full*ep/3600:5.2f} h'
              + ('   <-- exceeds 12 h session' if full*ep > 12*3600 else ''))
    json.dump({'limit': LIMIT, 'elapsed_s': EL, 'per_study_s': per,
               'full_epoch_s': full}, open('/kaggle/working/phase0_timing.json','w'), indent=2)